地名・日時指定型 天気予報プログラム
長谷川大祐　福田直生　坂井優希

地名と、０を今日として何日後の天気のデータが欲しいかを入力すると、その日にちのその場所の天気や気温などのデータを表示する

In [5]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode

In [6]:
%pip install geopy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:

from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="location_search_app")

def get_coordinates(place):
    location = geolocator.geocode(place, timeout=10)

    if location is not None:
        return location.latitude, location.longitude
    else:
        return None

# 地点を入力
place = input("地点を入力してください：")

# 緯度・経度を取得
coordinates = get_coordinates(place)

if coordinates is not None:
    lat, lon = coordinates
    print("緯度：", lat)
    print("経度：", lon)
else:
    print("地点が見つかりませんでした")


緯度： 34.6845445
経度： 135.8048359


In [8]:
number = int(input("何日後の天気を表示しますか？（0=今日）："))

# 過去何日分を含めるか
past_days = 0

# 取得する場所の現地時間を使う
timezone = "auto"

# 欲しい天気の項目を、カンマ区切りの文字列で指定する
daily_fields = (
    "weather_code,"
    "temperature_2m_max,temperature_2m_min,"
    "precipitation_probability_max,rain_sum,showers_sum"
)

# 通信を待つ最大秒数
timeout = 10

# ② 設定した変数をパラメータに渡し、URLを作る
url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": lat, "longitude": lon,
    "daily": daily_fields,
    "past_days": past_days,
    "timezone": timezone
})

# ③ APIから取得したJSONを、Pythonの辞書に変換する
with urlopen(url, timeout=timeout) as response:
    weather = json.load(response)

# ④ 日別データを表示する（セルの最後の値はprint不要）
def weather_name(code):
    if code == 0:
        return "快晴"
    elif code in [1, 2]:
        return "晴れ・一部曇り"
    elif code == 3:
        return "曇り"
    elif code in [45, 48]:
        return "霧"
    elif code in [51, 53, 55, 56, 57]:
        return "霧雨"
    elif code in [61, 63, 65, 66, 67]:
        return "雨"
    elif code in [71, 73, 75, 77, 85, 86]:
        return "雪"
    elif code in [80, 81, 82]:
        return "にわか雨"
    elif code in [95, 96, 99]:
        return "雷雨"
    else:
        return "不明"

if 0 <= number < len(weather["daily"]["time"]):
    print("天気：", weather_name(weather["daily"]["weather_code"][number]))
    print("地点：", place)
    print("日付：", weather["daily"]["time"][number])
    print("最高気温：", weather["daily"]["temperature_2m_max"][number], "℃")
    print("最低気温：", weather["daily"]["temperature_2m_min"][number], "℃")
    print("降水確率：", weather["daily"]["precipitation_probability_max"][number], "%")
    print("雨量：", weather["daily"]["rain_sum"][number], "mm")
    print("にわか雨：", weather["daily"]["showers_sum"][number], "mm")
else:
    print("指定された日付のデータはありません")

天気： 曇り
地点： 奈良市
日付： 2026-11-14
最高気温： 19.9 ℃
最低気温： 11.5 ℃
降水確率： 16 %
雨量： 0.0 mm
にわか雨： 0.0 mm
